In [22]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [23]:
from pathlib import Path

matches = list(Path("/kaggle/input").rglob("siamese-small.pt"))
path = matches[0].parent
config_path = list(Path("/kaggle/input").rglob("config.json"))[0]

In [24]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, AutoConfig

class SiameseClassifier(nn.Module):
    def __init__(self, config_path, num_labels=3):
        super().__init__()
        # self.encoder = AutoModel.from_pretrained(checkpoint, dtype=torch.float32)
        config = AutoConfig.from_pretrained(config_path, local_files_only=True)
        self.encoder = AutoModel.from_config(config)
        
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(2 * hidden, num_labels)

    def mean_pool(self, last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

    def forward(self, input_ids_a, attention_mask_a, input_ids_b, attention_mask_b):
        out_a = self.encoder(input_ids=input_ids_a, attention_mask=attention_mask_a).last_hidden_state
        out_b = self.encoder(input_ids=input_ids_b, attention_mask=attention_mask_b).last_hidden_state
        pool_a = self.mean_pool(out_a, attention_mask_a)
        pool_b = self.mean_pool(out_b, attention_mask_b)
        pooled = torch.cat([pool_a, pool_b], dim=-1)
        return {"logits": self.classifier(pooled)}

# ckpt = torch.load(f'{path}/siamese-small.pt', map_location="cpu")
ckpt = torch.load(path / "siamese-small.pt", map_location="cpu")
model = SiameseClassifier(config_path)
model.encoder.load_state_dict(ckpt["encoder_state"])
model.classifier.load_state_dict(ckpt["classifier_state"])
tokenizer = AutoTokenizer.from_pretrained(path / "siamese-small-tok")
model.eval()
print("loaded")

loaded


In [25]:
import json
import pandas as pd

test_df = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/test.csv")

def to_text(s):
    parts = json.loads(s)
    return " ".join(x if x is not None else "" for x in parts)

def preprocess_test(example):
    p = to_text(example["prompt"])
    a = to_text(example["response_a"])
    b = to_text(example["response_b"])
    enc_a = tokenizer(p, a, truncation=True, max_length=512)
    enc_b = tokenizer(p, b, truncation=True, max_length=512)
    return {
        "input_ids_a": enc_a["input_ids"],
        "attention_mask_a": enc_a["attention_mask"],
        "input_ids_b": enc_b["input_ids"],
        "attention_mask_b": enc_b["attention_mask"],
    }

from datasets import Dataset
test_ds = Dataset.from_pandas(test_df).map(preprocess_test, remove_columns=["prompt", "response_a", "response_b"])

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [26]:
import numpy as np
from scipy.special import softmax
from torch.utils.data import DataLoader

# reuse the SiameseCollator (returns labels too — make a test version without labels)
class SiameseTestCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer
    def __call__(self, features):
        a = self.tok.pad(
            [{"input_ids": f["input_ids_a"], "attention_mask": f["attention_mask_a"]} for f in features],
            return_tensors="pt",
        )
        b = self.tok.pad(
            [{"input_ids": f["input_ids_b"], "attention_mask": f["attention_mask_b"]} for f in features],
            return_tensors="pt",
        )
        return {
            "input_ids_a": a["input_ids"],
            "attention_mask_a": a["attention_mask"],
            "input_ids_b": b["input_ids"],
            "attention_mask_b": b["attention_mask"],
        }

collator = SiameseTestCollator(tokenizer)
loader = DataLoader(test_ds, batch_size=32, collate_fn=collator)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

all_probs = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch)["logits"]
        probs = softmax(logits.float().cpu().numpy(), axis=-1)
        all_probs.append(probs)

probs = np.concatenate(all_probs, axis=0)

In [27]:
sub = pd.DataFrame({
    "id": test_df["id"],
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_tie":     probs[:, 2],
})
sub.to_csv("/kaggle/working/submission.csv", index=False)